<a href="https://colab.research.google.com/github/zohaib-mzg/Flyrank-ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4: Baseline Action Score and Top 10 Review

Same lane, Refresh / Content Opportunity Scoring, locked in for this week. Before building anything I checked two signals my rule idea leans on, then encoded one rule the way the live session built one, then read my own top ten with a skeptic's eye.

In [1]:
import os, sys, subprocess
import pandas as pd

REPO_URL = "https://github.com/zohaib-mzg/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../..')
elif not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

print('Working dir:', os.getcwd())
assert os.path.exists('data/raw/content_refresh_anonymized.csv'), 'starter CSV not found'

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f"{len(df):,} rows loaded")

Working dir: /content/Flyrank-ML-Internship
30,000 rows loaded


## 1. Two signal checks

### Signal 1, staleness, behind the refresh flags

My hypothesis was that staler pages, further from their last update, are more likely to be declining. That's the assumption sitting underneath every stale visible page flag.

In [2]:
staleness_check = df.groupby('freshness_tier').agg(
    n=('content_id', 'size'),
    decline_rate=('trend_direction', lambda s: (s == 'down').mean()),
).reindex(['0-30', '31-90', '91-180', '181+'])

staleness_check

,n,decline_rate
freshness_tier,,
0-30,20480,0.511377
31-90,175,0.588571
91-180,9171,0.611057
181+,174,0.471264


Verdict: MIXED.

The decline rate rises from 51.1 percent in the 0 to 30 day tier to 58.9 percent in 31 to 90 days, then peaks at 61.1 percent in 91 to 180 days, so far that fits the hypothesis. But the stalest tier, 181 or more days since update, drops back down to 47.1 percent, the lowest decline rate of all four tiers, on 174 pages. If staleness alone drove decline, this tier should be the worst one, not one of the best.

I'm treating this as a genuine negative rather than a data problem. It tells me staleness on its own isn't a safe single signal for a refresh rule. A page can sit untouched for over six months and still be stable or growing, probably because it never needed touching in the first place. I'm not building my rule on this signal alone. It stays as context, not as the driver.

### Signal 2, CTR versus position, behind the CTR fix logic

My hypothesis here was that pages in stronger position tiers should show meaningfully higher average CTR than pages in weaker tiers. That relationship is what makes a low CTR at a strong position an anomaly worth flagging instead of just noise.

In [3]:
position_check = df.groupby('position_tier').agg(
    n=('content_id', 'size'),
    avg_ctr=('ctr', 'mean'),
).reindex(['top_3', 'page_1', 'striking', 'page_3_5', 'deep'])

position_check

,n,avg_ctr
position_tier,,
top_3,2321,1.483611
page_1,11814,0.652467
striking,7304,0.323239
page_3_5,7242,0.222484
deep,1319,0.150212


Verdict: CONFIRMED.

Average CTR falls in the order I'd expect as position gets weaker. Top_3 is highest, then page_1, then striking, then page_3_5, then deep is lowest, with n ranging from 1,319 to 11,814 pages per tier. The relationship holds cleanly enough that a page sitting in a strong tier with a CTR far below its tier's average really is a genuine anomaly, which is exactly the assumption the low CTR visible page flag depends on.

One thing worth being honest about: top_3's average CTR of 1.48 is inflated by a handful of outlier pages with unusually high values, since a real CTR shouldn't exceed 1. I didn't clean that outlier out for this check because the ordering across tiers still holds regardless, but it means any score I build using the top_3 tier average as a benchmark will look bigger than it should for pages in that tier specifically. I come back to this in Section 4.

### The rule I ended up encoding

Because Signal 1 came back mixed and Signal 2 came back confirmed, I built this week's rule entirely on Signal 2, not on staleness. Reason code: low_ctr_visible_page, the exact threshold from the lane guide, impressions_90d at least 500, avg_position between 0 and 20, ctr under 0.5. Score: for each qualifying page, the peer tier's average ctr minus that page's actual ctr, multiplied by impressions_90d, basically an estimate of clicks left on the table if the page's CTR matched its own tier's average. Action label: ctr_review. Nothing here touches trend_direction or any future window, only CTR, position, and impression volume, all observable right now.

## 2. The rule and the ranked queue

In [4]:
peer_tier_avg_ctr = df.groupby('position_tier')['ctr'].transform('mean')

qualifies = (
    (df['impressions_90d'] >= 500)
    & (df['avg_position'] > 0) & (df['avg_position'] <= 20)
    & (df['ctr'] < 0.5)
)

queue = df[qualifies].copy()
queue['peer_tier_avg_ctr'] = peer_tier_avg_ctr[qualifies]
queue['score'] = (queue['peer_tier_avg_ctr'] - queue['ctr']) * queue['impressions_90d']
queue['reason_code'] = 'low_ctr_visible_page'
queue['action'] = 'ctr_review'

queue = queue.sort_values('score', ascending=False).reset_index(drop=True)

print(f"{len(queue):,} pages qualify for low_ctr_visible_page out of {len(df):,} total")

output_cols = ['content_id', 'client_id', 'score', 'reason_code', 'action',
               'avg_position', 'position_tier', 'ctr', 'peer_tier_avg_ctr',
               'impressions_90d', 'trend_direction']

os.makedirs('work/outputs', exist_ok=True)
queue[output_cols].to_csv('work/outputs/baseline_action_score.csv', index=False)
print("written to work/outputs/baseline_action_score.csv")

queue[output_cols].head(10)

9,759 pages qualify for low_ctr_visible_page out of 30,000 total
written to work/outputs/baseline_action_score.csv


,content_id,client_id,score,reason_code,action,avg_position,position_tier,ctr,peer_tier_avg_ctr,impressions_90d,trend_direction
0,content_8c19996aa890,client_4e07408562,679143.820819,low_ctr_visible_page,ctr_review,2.5,top_3,0.15,1.483611,509252,down
1,content_4c36c775b818,client_4e07408562,497192.249268,low_ctr_visible_page,ctr_review,2.3,top_3,0.41,1.483611,463103,down
2,content_8451fc6f034d,client_d029fa3a95,395591.379371,low_ctr_visible_page,ctr_review,2.3,top_3,0.03,1.483611,272144,up
3,content_5fe46e04994d,client_4e07408562,265311.627747,low_ctr_visible_page,ctr_review,4.2,page_1,0.14,0.652467,517715,down
4,content_e12868d1f396,client_4e07408562,211634.457079,low_ctr_visible_page,ctr_review,2.9,top_3,0.07,1.483611,149712,stable
5,content_aaef01a50def,client_19581e27de,208119.083008,low_ctr_visible_page,ctr_review,5.4,page_1,0.25,0.652467,517109,stable
6,content_4a6607efcb46,client_6208ef0f77,188722.351142,low_ctr_visible_page,ctr_review,2.2,top_3,0.01,1.483611,128068,up
7,content_36ff89c8214e,client_19581e27de,177786.075959,low_ctr_visible_page,ctr_review,7.3,page_1,0.05,0.652467,295097,stable
8,content_1a9e894be2e2,client_19581e27de,175822.135060,low_ctr_visible_page,ctr_review,4.0,page_1,0.23,0.652467,416180,down
9,content_db5989a78dd3,client_4e07408562,152700.078746,low_ctr_visible_page,ctr_review,5.4,page_1,0.21,0.652467,345111,up


## 3. Top 10 review

For each page below, one line covers the action, why it's there, and what would make it wrong.

1. content_8c19996aa890, ctr_review. Sitting at position 2.5 with a 0.15 CTR against a roughly 1.48 tier average, on 509,252 impressions, so the gap is large on real volume. This would be the wrong call if the SERP shows a featured snippet or a People Also Ask box that's structurally stealing clicks no title or meta rewrite could recover.
2. content_4c36c775b818, ctr_review. Position 2.3, CTR 0.41, already close to a typical CTR. Its high score mostly comes from 463,103 impressions and the inflated top_3 benchmark, not a genuinely alarming gap. Wrong if the true opportunity here is much smaller than the score implies, see Section 4.
3. content_8451fc6f034d, ctr_review. Position 2.3, CTR just 0.03 on 272,144 impressions, one of the largest real gaps in the whole queue. Wrong if this page's true intent is informational and a low click rate is simply expected here, for example if users already get their answer straight from the snippet.
4. content_5fe46e04994d, ctr_review. Position 4.2, CTR 0.14 against a page_1 average of 0.65, on 517,715 impressions. Wrong if the title already describes low intent content accurately and a higher CTR would just mean more bounces.
5. content_e12868d1f396, ctr_review. Position 2.9, CTR 0.07, trend stable, on 149,712 impressions. Wrong if this page recently changed titles and CTR simply hasn't caught up yet, meaning the fix is already in motion.
6. content_aaef01a50def, ctr_review. Position 5.4, CTR 0.25 against a page_1 average of 0.65, a real but moderate gap on 517,109 impressions. Wrong if the query mix behind this page is mostly navigational, where a lower CTR is normal and not something to fix.
7. content_4a6607efcb46, ctr_review. Position 2.2, CTR just 0.01, the lowest raw CTR in the top ten, on 128,068 impressions, trend up. Wrong if the low CTR is an artifact of a very recent ranking jump and clicks simply haven't caught up to the new position yet.
8. content_36ff89c8214e, ctr_review. Position 7.3, CTR 0.05 against a page_1 average of 0.65, on 295,097 impressions. Wrong if the ranking query is broad and low intent, so most of the tier's impressions come from a different kind of search than this page actually serves.
9. content_1a9e894be2e2, ctr_review. Position 4.0, CTR 0.23, trend down, on 416,180 impressions. Wrong if the CTR problem is really secondary to the decline itself, meaning the real fix is content relevance, not a title or meta change.
10. content_db5989a78dd3, ctr_review. Position 5.4, CTR 0.21, trend up, on 345,111 impressions. Wrong if this page is already trending up on its own and a CTR intervention risks disrupting something that's already working.

## 4. Weak picks

Rows 2 and 6 are the weakest picks in this top ten. Both have a CTR, 0.41 and 0.25, that isn't dramatically low on its own, closer to what I'd expect from a typical page, and both are ranked as high as they are mainly because of one shared issue. The peer tier average CTR I'm subtracting against is inflated, top_3 by outlier pages with a CTR above 1, and page_1 to a lesser degree by the same kind of skew. That means the scores for these two rows probably overstate the real opportunity, and I wouldn't put them ahead of rows with a genuinely large CTR gap, like row 3 or row 7, just because the raw score says so.

If I revisit this rule later, the fix is to use the tier's median CTR instead of its mean as the benchmark, since a median is far less sensitive to a few outlier pages pulling the whole tier average upward.

## 5. Self-check

I checked two signals with visible bucket tables and n before writing any rule, not after. One came back mixed, staleness, and I didn't build the rule on it. One came back confirmed, CTR versus position, and that's what the rule actually uses.

One rule encoded: low_ctr_visible_page, one reason code, one action label, ctr_review, and a score with a clear real world meaning, estimated clicks left on the table.

The ranked queue is written from the notebook to work/outputs/baseline_action_score.csv and isn't committed, which matches the CI leak guard rule since it regenerates from source data on every run.

Ten rows reviewed individually, action, why, and what would make it wrong, and two of the ten flagged honestly as weaker picks than their score suggests, with the specific reason named, an inflated peer benchmark, not just a vague hedge.

No future window or label derived inputs show up anywhere in the rule. The score only uses CTR, position, and impression volume, all observable at the decision moment. trend_direction appears in the output table for context only, and never inside the score itself.